In [1]:
!pip install -q kaggle xgboost seaborn joblib


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [4]:
!mkdir -p ~/.kaggle
!cp /workspaces/codespaces-blank/zd-nids/notebooks/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [6]:
!kaggle datasets list

ref                                                                  title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-------------------------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
amar5693/screen-time-sleep-and-stress-analysis-dataset               Screen Time, Sleep & Stress Analysis Dataset            787136  2026-02-13 06:56:18.757000           6415        127                1  
amar5693/student-performance-dataset                                 Student Performance Dataset                             177286  2026-02-12 06:04:44.613000           5554         90                1  
algozee/heart-decices                                                Heart Disease Prediction Using Machine Learning       10070453  2026-02-22 17:32:58.757000            809      

In [ ]:
!kaggle datasets download -d chethuhn/network-intrusion-dataset
!unzip network-intrusion-dataset.zip

Dataset URL: https://www.kaggle.com/datasets/chethuhn/network-intrusion-dataset
License(s): CC0-1.0
100%|████████████████████████████████████████| 230M/230M [00:16<00:00, 14.5MB/s]

Archive:  network-intrusion-dataset.zip
  inflating: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv  
  inflating: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv  
  inflating: Friday-WorkingHours-Morning.pcap_ISCX.csv  
  inflating: Monday-WorkingHours.pcap_ISCX.csv  
  inflating: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv  
  inflating: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv  
  inflating: Tuesday-WorkingHours.pcap_ISCX.csv  
  inflating: Wednesday-workingHours.pcap_ISCX.csv  


In [ ]:
!kaggle datasets download -d solarmainframe/ids-intrusion-csv

In [1]:
!unzip ids-intrusion-csv.zip

Archive:  ids-intrusion-csv.zip
  inflating: 02-14-2018.csv          
  inflating: 02-15-2018.csv          
  inflating: 02-16-2018.csv          
  inflating: 02-20-2018.csv          
  inflating: 02-21-2018.csv          
  inflating: 02-22-2018.csv          
  inflating: 02-23-2018.csv          
  inflating: 02-28-2018.csv          
  inflating: 03-01-2018.csv          
  inflating: 03-02-2018.csv          


In [ ]:
import pandas as pd
import glob
import os

# Folder where original CICIDS files are present
data_folder = "/workspaces/codespaces-blank/zd-nids/notebooks"   # change if needed

# Get only original dataset files
files = glob.glob(os.path.join(data_folder, "*.csv"))

# Remove previously combined file if exists
output_file = "cicids2018_combined.csv"
if os.path.exists(output_file):
    os.remove(output_file)

print("Files found:")
for f in files:
    print(f)

first = True

for file in files:
    print("Processing:", os.path.basename(file))
    
    for chunk in pd.read_csv(file, chunksize=100000, low_memory=False):
        # Clean column names
        chunk.columns = chunk.columns.str.strip()
        
        # Write to combined file
        chunk.to_csv(
            output_file,
            mode='w' if first else 'a',
            index=False,
            header=first
        )
        first = False

print("\nCombined dataset saved:", output_file)

Files found:
/workspaces/codespaces-blank/zd-nids/notebooks/02-23-2018.csv
/workspaces/codespaces-blank/zd-nids/notebooks/03-01-2018.csv
/workspaces/codespaces-blank/zd-nids/notebooks/02-16-2018.csv
/workspaces/codespaces-blank/zd-nids/notebooks/02-22-2018.csv
/workspaces/codespaces-blank/zd-nids/notebooks/02-20-2018.csv
/workspaces/codespaces-blank/zd-nids/notebooks/03-02-2018.csv
/workspaces/codespaces-blank/zd-nids/notebooks/02-21-2018.csv
/workspaces/codespaces-blank/zd-nids/notebooks/02-14-2018.csv
/workspaces/codespaces-blank/zd-nids/notebooks/cicids2018_combined.csv
/workspaces/codespaces-blank/zd-nids/notebooks/02-28-2018.csv
/workspaces/codespaces-blank/zd-nids/notebooks/02-15-2018.csv
Processing: 02-23-2018.csv
Processing: 03-01-2018.csv
Processing: 02-16-2018.csv
Processing: 02-22-2018.csv
Processing: 02-20-2018.csv


In [2]:
df_check = pd.read_csv("cicids2018_combined.csv", nrows=200000)
df_check.columns = df_check.columns.str.strip()

print("Number of classes:", df_check["Label"].nunique())
print("Labels:", df_check["Label"].unique())

Number of classes: 2
Labels: ['BENIGN' 'Infiltration']


In [2]:
import numpy as np
import pandas as pd

print("Loading dataset...")

# Load manageable size
df = pd.read_csv("cicids2017_combined.csv", nrows=150000, low_memory=False)

# Clean column names
df.columns = df.columns.str.strip()

# Remove invalid values
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

print("Dataset shape:", df.shape)

Loading dataset...
Dataset shape: (149921, 79)


In [3]:
from sklearn.preprocessing import LabelEncoder

# Target (MULTI-CLASS)
y = df["Label"]

# Features
X = df.drop(columns=["Label"])
X = X.select_dtypes(include=[np.number])

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Number of classes:", len(le.classes_))

Number of classes: 2
